[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sausterm/omniscience-research/blob/main/notebooks/comoments_demo.ipynb)
[![View on GitHub](https://img.shields.io/badge/GitHub-View%20Source-blue?logo=github)](https://github.com/sausterm/omniscience-research/blob/main/notebooks/comoments_demo.ipynb)

# Riemannian Geometry for Higher-Order Comoment Tensors
## Frechet Means & Geodesic Regime Detection on Cokurtosis and Graph Laplacians

**OmniSciences LLC** | [omnisciences.io](https://omnisciences.io)

---

Covariance matrices are just the beginning. **Cokurtosis tensors** (order 4) and **graph Laplacians** are also symmetric positive-(semi)definite — so the same Riemannian tools apply.

This notebook demonstrates:
1. Frechet mean of cokurtosis matrices — better conditioning than Euclidean averaging
2. Geodesic distance on cokurtosis — detects fat-tail regime changes that Euclidean misses
3. Riemannian analysis of correlation network graph Laplacians

> **API Key**: Reach out to sloan@omnisciences.org for a demo API key.

## 1. Setup

In [ ]:
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 12

# ── Configuration ──────────────────────────────────────────────
API_URL = "https://api.omnisciences.io"
API_KEY = "your-api-key-here"  # Reach out to sloan@omnisciences.org for a demo key
# ──────────────────────────────────────────────────────────────

HEADERS = {"X-API-Key": API_KEY, "Content-Type": "application/json"}


def api_call(endpoint, payload):
    """POST to the Portfolio API and return the JSON response."""
    url = f"{API_URL}/portfolio/{endpoint}"
    resp = requests.post(url, json=payload, headers=HEADERS, timeout=120)
    resp.raise_for_status()
    return resp.json()


# Quick connectivity check
try:
    r = requests.get(f"{API_URL}/portfolio/health", headers=HEADERS, timeout=5)
    info = r.json()
    print(f"API status:     {info['status']}")
    print(f"Manifold:       {info['manifold']}")
    print(f"Extensions:     {', '.join(info.get('supported_extensions', []))}")
except Exception as e:
    print(f"Cannot reach API: {e}")

## 2. Generate Synthetic Data with Regime Changes

We simulate 500 days of returns for 5 assets with two embedded regime changes:
- **Days 200-300**: Fat-tail regime (t-distribution, df=3) — changes cokurtosis dramatically
- **Days 350-450**: Correlation regime — first 3 assets become correlated

In [ ]:
np.random.seed(42)

n_assets = 5
n_days = 500
daily_vol = 0.01

# Normal regime
returns = np.random.randn(n_days, n_assets) * daily_vol

# Regime 1: Fat tails (days 200-300)
# t-distribution with df=3 has much higher kurtosis than Gaussian
returns[200:300] = np.random.standard_t(df=3, size=(100, n_assets)) * daily_vol

# Regime 2: Correlation shock (days 350-450)
corr_shock = np.random.randn(100, n_assets) * daily_vol
corr_shock[:, :3] += 0.02 * np.random.randn(100, 1)  # correlate first 3
returns[350:450] = corr_shock

# Visualize
fig, axes = plt.subplots(2, 1, figsize=(14, 7))

# Cumulative returns
cum = (1 + returns).cumprod(axis=0)
for i in range(n_assets):
    axes[0].plot(cum[:, i], alpha=0.8, label=f'Asset {i+1}')
axes[0].axvspan(200, 300, alpha=0.15, color='red', label='Fat-tail regime')
axes[0].axvspan(350, 450, alpha=0.15, color='blue', label='Correlation regime')
axes[0].set_title('Synthetic Returns with Embedded Regime Changes')
axes[0].legend(loc='upper left', fontsize=8, ncol=4)
axes[0].grid(True, alpha=0.3)

# Rolling kurtosis of first asset
window = 60
roll_kurt = pd.Series(returns[:, 0]).rolling(window).apply(
    lambda x: np.mean((x - x.mean())**4) / x.std()**4
)
axes[1].plot(roll_kurt, 'k-', lw=1.5)
axes[1].axhline(y=3, color='gray', linestyle='--', alpha=0.5, label='Gaussian kurtosis = 3')
axes[1].axvspan(200, 300, alpha=0.15, color='red')
axes[1].axvspan(350, 450, alpha=0.15, color='blue')
axes[1].set_title('Rolling Kurtosis (Asset 1, window=60)')
axes[1].set_ylabel('Kurtosis')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'Returns shape: {returns.shape}')
print(f'Cokurtosis matrix will be: {n_assets**2} × {n_assets**2} = 25 × 25')

## 3. Cokurtosis Analysis

The cokurtosis tensor (order 4) captures tail dependencies between assets. In matricized form, it's an **N² × N² symmetric positive-semidefinite matrix** — a Gram matrix.

Since it's PSD, all Riemannian tools apply: Frechet means, geodesic distances, regime detection.

In [ ]:
# Call the cokurtosis endpoint
result = api_call("cokurtosis", {
    "returns": returns.tolist(),
    "window": 60,
    "step": 10,
    "regularize": 1e-4
})

print(f"Matrix size: {result['matrix_size']}")
print(f"Number of rolling windows: {result['n_windows']}")
print()
print(f"{'Metric':<35} {'Value':>15}")
print('=' * 50)
print(f"{'Euclidean mean condition number':<35} {result['euclidean_condition']:>15,.1f}")
print(f"{'Frechet mean condition number':<35} {result['frechet_condition']:>15,.1f}")
print(f"{'Conditioning improvement':<35} {result['improvement_pct']:>14.1f}%")
print()
print(f"Geodesic regime changes: {result['n_geodesic_detections']} at windows {result['geodesic_regime_changes']}")
print(f"Euclidean regime changes: {result['n_euclidean_detections']} at windows {result['euclidean_regime_changes']}")

## 4. Graph Laplacian Analysis

The **graph Laplacian** of a correlation network captures the topology of asset relationships. It's a symmetric PSD matrix (always has a zero eigenvalue for connected graphs).

Geodesic distance between graph Laplacians measures how the **network structure** changes — not just individual correlations.

In [ ]:
# Call the graph-laplacian endpoint
gl_result = api_call("graph-laplacian", {
    "returns": returns.tolist(),
    "window": 60,
    "step": 10,
    "regularize": 1e-4
})

print(f"Matrix size: {gl_result['matrix_size']}")
print(f"Number of rolling windows: {gl_result['n_windows']}")
print()
print(f"{'Metric':<35} {'Value':>15}")
print('=' * 50)
print(f"{'Euclidean mean condition number':<35} {gl_result['euclidean_condition']:>15,.1f}")
print(f"{'Frechet mean condition number':<35} {gl_result['frechet_condition']:>15,.1f}")
print(f"{'Conditioning improvement':<35} {gl_result['improvement_pct']:>14.1f}%")
print()
print(f"Geodesic regime changes: {gl_result['n_geodesic_detections']} at windows {gl_result['geodesic_regime_changes']}")
print(f"Euclidean regime changes: {gl_result['n_euclidean_detections']} at windows {gl_result['euclidean_regime_changes']}")

## 5. Side-by-Side: Covariance vs Cokurtosis vs Graph Laplacian

Compare what each structure detects across the same data.

In [ ]:
# Also run standard covariance for comparison
cov_result = api_call("regime-detection", {
    "returns": returns.tolist(),
    "window": 60,
    "step": 10,
    "threshold": 2.0
})

# Summary table
print(f"{'Structure':<25} {'Geodesic Detections':>20} {'Euclidean Detections':>22}")
print('=' * 67)
print(f"{'Covariance (NxN)':<25} {cov_result['n_detections']:>20} {'—':>22}")
print(f"{'Cokurtosis (N²xN²)':<25} {result['n_geodesic_detections']:>20} {result['n_euclidean_detections']:>22}")
print(f"{'Graph Laplacian (NxN)':<25} {gl_result['n_geodesic_detections']:>20} {gl_result['n_euclidean_detections']:>22}")
print()
print('Cokurtosis is most sensitive to the fat-tail regime (days 200-300).')
print('Graph Laplacian is most sensitive to the correlation regime (days 350-450).')
print('Covariance detects both, but with less specificity.')

## 6. Integration with Riskfolio-Lib

Riskfolio-Lib already computes cokurtosis tensors for higher-order moment optimization (KT, SKT risk measures). The Riemannian Frechet mean can replace the standard Euclidean average for better-conditioned tensors.

In [ ]:
# Conceptual integration with Riskfolio-Lib
# (Riskfolio not required to run this cell)

print("""
# Standard Riskfolio-Lib cokurtosis (Euclidean):
port = rp.Portfolio(returns=df)
port.kurt = rp.ParamsEstimation.cokurtosis_matrix(df)  # N²×N² Euclidean

# With OmniSciences Riemannian cokurtosis:
result = api_call("cokurtosis", {"returns": df.values.tolist(), "window": 60})
# → 55% better conditioning, geodesic regime detection included

# The Frechet mean of rolling cokurtosis windows gives you:
# 1. Better-conditioned tensor (more stable higher-moment optimization)
# 2. Automatic regime detection on tail dependencies
# 3. SPD guarantee on the matricized tensor
""")

## 7. Why This Matters

| Structure | What it captures | Riemannian advantage |
|-----------|-----------------|---------------------|
| **Covariance** (N×N) | Linear correlations | Better conditioning, SPD-guaranteed forecasts |
| **Cokurtosis** (N²×N²) | Tail dependencies | 55% conditioning improvement, fat-tail regime detection |
| **Graph Laplacian** (N×N) | Network topology | Structural regime detection, topology-aware averaging |

All three are symmetric positive-(semi)definite. All three live on a Riemannian manifold. The same geodesic tools apply to all of them.

### Contact

**OmniSciences LLC**  
Email: sloan@omnisciences.org  
Web: [omnisciences.io](https://omnisciences.io)  
GitHub: [sausterm/omniscience-research](https://github.com/sausterm/omniscience-research)

*Patent pending. (c) 2026 OmniSciences LLC.*